In [ ]:
import healpy as hp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pathlib import Path

from dusty_colors import use_matplotlib_style

use_matplotlib_style()

In [ ]:
sample = "dp1_default"
save_figs = True

In [ ]:
sample_dir = Path(f"../results/samples/{sample}")

# The footprint table is one row per selected galaxy, carrying only positions,
# field, HEALPix pixel, and jackknife region. Jackknife regions are assigned at
# the sample stage, so this is the only file that has them.
fp = pd.read_parquet(sample_dir / "footprint.parquet")

NSIDE = 1024
area_per_pix = hp.nside2pixarea(NSIDE, degrees=True)
print(f"Total area: {fp.pixel.nunique() * area_per_pix:.2f} deg^2")
for field, pixels in fp.groupby("field").pixel:
    print(f"  {field}: {pixels.nunique() * area_per_pix:.2f} deg^2")

# Regions are numbered globally across fields (field 0 gets 0-2, field 1 gets
# 3-5, ...), so reducing modulo the per-field count gives the within-field
# sector index and keeps every panel using the same three colors.
regions_per_field = fp.jackknife_region.nunique() // fp.field.nunique()


# Now plot the jackknife regions
def plot_field(ax, field):
    ax.set_title(field)
    ax.set_xlabel("RA")
    ax.set_ylabel("Dec")
    ax.invert_xaxis()

    sub = fp.query(f"field == '{field}'")

    # A degree of RA spans only cos(dec) degrees on the sky, so scale the aspect
    # ratio instead of using "equal". The fields run from Dec +7 to -49, and
    # without this the southern ones are drawn stretched in RA.
    ax.set_aspect(1 / np.cos(np.deg2rad(sub.dec.mean())))

    ax.scatter(
        sub.ra,
        sub.dec,
        s=0.1,
        c=[f"C{i % regions_per_field}" for i in sub.jackknife_region],
        rasterized=True,
    )

    # Label each region with its global (1-based) number at its median position.
    # The regions are angular sectors about the field center, so the median lands
    # inside the sector it labels.
    for i in np.sort(sub.jackknife_region.unique()):
        sub2 = sub.query(f"jackknife_region == {i}")
        x = np.median(sub2.ra)
        y = np.median(sub2.dec)
        ax.text(x, y, str(i + 1), ha="center", va="center", fontsize=12, color="k")


fields = ["ECDFS", "EDFS", "Rubin SV 38 7", "Rubin SV 95 -25"]

fig, axes = plt.subplots(
    1, len(fields), figsize=(9, 2.2), dpi=150, constrained_layout=True
)
for ax, field in zip(axes, fields):
    plot_field(ax, field)

if save_figs:
    fig.savefig("../figures/jackknife_regions.pdf", bbox_inches="tight")